# Global Maps and Tabels

This notebook demonstrates preprocessing and plotting of the global maps displayed in Fig. 1, 2 & 4 as well as Supplementary Fig. S2-S10, and the computations for the Supllementary Table S4.

### Note

To replicate the results presented in the publication, it is essential to preprocess the complete ESM output data as described in the README or Method section of the publication.

#### Import Libraries

In [ ]:
# ========== Import Required Libraries ==========
import sys
import dask
from dask.diagnostics import ProgressBar
import numpy as np
import xarray as xr
import copy

In [ ]:
# ========== Configure Paths ==========
# Define the full path to the directories containing utility scripts and configurations
sys.path.append('../../src')
sys.path.append('../../src/data_handling')
sys.path.append('../../src/analysis')
sys.path.append('../../src/visualization')

# Import custom utility functions and configurations
import load_data as load_dat
import process_data as pro_dat
import compute_statistics as comp_stats
import global_maps_and_tabels as glob_map_tab

#import data directory
from config import DATA_DIR

In [ ]:
# ========== Define Font Style ==========
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Nimbus Sans'

### Step 1: Load Data

In [ ]:
# Load CMIP6 BGWS and BGWS variables
masked_ds_dict, masked_ds_dict_diff, hydro_mask = comp_stats.bgws_period_stats()

In [ ]:
# Load observation-based and ERA5 land reference data
ds_ref_mean, ds_ref_clim = comp_stats.load_reference_mean_and_clim(hydro_mask=hydro_mask)
ds_early, ds_late, ds_change = comp_stats.load_historical_15yr_change_for_benchmark(hydro_mask=hydro_mask)

In [ ]:
# Load CMIP6 hydroecological and climatic data
masked_ds_dict_driver, masked_ds_dict_diff_driver, _ = comp_stats.driver_period_stats(hydro_mask=hydro_mask)

In [ ]:
# Combine data for plotting
combined_dict = {}
for experiments in masked_ds_dict.keys():
    combined_dict[experiments] = masked_ds_dict[experiments]
for experiments in combined_dict.keys():
    for model in combined_dict[experiments].keys():
        if model ==  '12 model ensemble std' or model == 'OBS' or model == 'ERA5_land':
            pass
        else:
            for var in masked_ds_dict_driver[experiments][model].data_vars:
                combined_dict[experiments][model][var] = masked_ds_dict_driver[experiments][model][var]

combined_dict_diff = {}
for experiments in masked_ds_dict_diff.keys():
    combined_dict_diff[experiments] = masked_ds_dict_diff[experiments]
    for model in combined_dict_diff[experiments].keys():
        if model ==  '12 model ensemble std':
            pass
        else:
            for var in masked_ds_dict_diff_driver[experiments][model].data_vars:
                combined_dict_diff[experiments][model][var] = masked_ds_dict_diff_driver[experiments][model][var]

### Step 2: Plot Fig. 1

In [ ]:
glob_map_tab.figure1(
    combined_dict,
    ds_ref_mean,
    dpi=300,
    filetype='png',
    savepath=None
)

### Step 3: Plot Fig. 2

In [ ]:
glob_map_tab.figure2(
    masked_ds_dict['historical'],
    masked_ds_dict_diff,
    min_agree_models=8,
    dpi=300,
    filetype="png",
    savepath=None
)

### Step 4: Plot Fig. 4

In [ ]:
fig = glob_map_tab.figure4_5_combined(
    ds_dict=masked_ds_dict_diff,
    model="12 model ensemble mean",
    period="ssp370_ff-historical",
    threshold_bgws=0,
    threshold_change=0,
    magnitude_source="joint",
    min_agree_models=8,
    agreement_mode="mask", 
    dpi=300,
    filetype="png",
    savepath=None
)

### Step 5: Plot historical change benchmarking (Figs. S2-S5)

In [ ]:
# Combine historical period mean data for plotting
ds_dict_hist_bench_mean = {}
ds_dict_hist_bench_mean = masked_ds_dict["historical"]
for model in ds_ref_mean["historical"].keys():
    ds_dict_hist_bench_mean[model] = ds_ref_mean["historical"][model]

In [ ]:
# Plot Fig. S2
fig, out = glob_map_tab.plot_bgws_validation_figure(
    data_dict=ds_dict_hist_bench_mean,
    var_name="bgws_tran_mean",
    mode="state",
    eps=.5,
    vmin=-60,
    vmax=60,
    map_bin_width=5,
    cbar_tick_step=10,
    cbar_label=r"BGWS [%]",
    scatter_ylabel=r"CMIP6 ensemble mean BGWS [%]",
    scatter_xlabel_era=r"ERA5-Land BGWS [%]",
    scatter_xlabel_obs=r"Obs.-based BGWS [%]",
    scatter_percentile_range=(1, 99),
    savepath=None,
)

In [ ]:
# Plot Fig. S3
fig, out = glob_map_tab.plot_bgws_validation_figure(
    data_dict=ds_change["historical"],
    var_name="bgws_tran_mean",
    mode="change",
    eps=.5,
    vmin=-6,
    vmax=6,
    map_bin_width=1,
    cbar_tick_step=2,
    cbar_label=r"$\Delta$BGWS (2000-2014 minus 1985-1999) [ppts]",
    scatter_ylabel=r"CMIP6 ensemble mean $\Delta$BGWS [ppts]",
    scatter_xlabel_era=r"ERA5-Land $\Delta$BGWS [ppts]",
    scatter_xlabel_obs=r"Obs.-based $\Delta$BGWS [ppts]",
    scatter_percentile_range=(1, 99),
    savepath=None,
)

In [ ]:
# Plot Fig. S4
fig, out = glob_map_tab.plot_partitioning_validation_figure(
    data_dict=ds_dict_hist_bench_mean,
    var_name="et_over_p_mean",
    vmin=0,
    vmax=60,
    map_bin_width=5,
    cbar_tick_step=10,
    cbar_label=r"$E_t/P$ [%]",
)

In [ ]:
# Plot Fig. S5
fig, out = glob_map_tab.plot_partitioning_validation_figure(
    data_dict=ds_dict_hist_bench_mean,
    var_name="r_over_p_mean",
    vmin=0,
    vmax=60,
    map_bin_width=5,
    cbar_tick_step=10,
    cbar_label=r"$R/P$ [%]",
)

### Step 6: Plot historical climatic and hydroecological states (Figs. S6 & S7)

In [ ]:
# Plot Fig. S6
fig = glob_map_tab.plot_historical_hydroeco_ensemble_maps(
    data=combined_dict["historical"],  
    ens_key="12 model ensemble mean",
    savepath=None
)

In [ ]:
# Plot Fig. S7
fig = glob_map_tab.plot_rx5day_ratio_map_and_scatters(
    masked_ds_dict["historical"],
    ens_key="12 model ensemble mean",
    ratio_var="rx5day_ratio",
    pr_var="pr_mean",
    bgws_var="bgws_tran_mean",
    figsize=(14, 9.5),
)

### Step 7: Plot Inter-model uncertainty in BGWS (Fig. S8)

In [ ]:
fig = glob_map_tab.plot_bgws_ensemble_std_two_panel(
    historical_dict=masked_ds_dict["historical"],
    change_dict=combined_dict_diff["ssp370_ff-historical"],
    hist_key="12 model ensemble std",
    change_key="12 model ensemble std",
    var_name="bgws_tran_mean",
    figsize=(14, 5.8),
    dpi=300,
    savepath=None
)

### Step 8: Projected BGWS regime shifts (Fig. S9)

In [ ]:
fig = glob_map_tab.plot_bgws_regime_shift_figure(
    ds_dict=masked_ds_dict,
    ds_dict_change=combined_dict_diff,
    historical_period="historical",
    future_period="ssp370_ff",
    change_period="ssp370_ff-historical",
    hist_key="12 model ensemble mean",
    future_key="12 model ensemble mean",
    change_key="12 model ensemble mean",
    var_name="bgws_tran_mean",
    min_agree_models=8,
    stipple_step=1,
    stipple_size=12,
    # savepath="fig_bgws_shift.png",
    # dpi=300,
)

### Step 9: Projected changes in climatic and hydroecological variables (Fig. S10)

In [ ]:
# Compute relative change for P, R and E_t 
combined_dict_diff_rel = copy.deepcopy(combined_dict_diff)

rel_change_dict = glob_map_tab.build_relative_change_dict(
    masked_ds_dict=masked_ds_dict,
    historical_key="historical",
    future_key="ssp370_ff",
    ens_key="12 model ensemble mean",
    variables=("pr_mean", "mrro_mean", "tran_mean"),
    out_key="ssp370_ff-historical_rel",
)

combined_dict_diff_rel.update(rel_change_dict)

In [ ]:
fig = glob_map_tab.plot_change_hydroeco_ensemble_maps(
    data=combined_dict_diff["ssp370_ff-historical"],
    rel_change_data=combined_dict_diff_rel["ssp370_ff-historical_rel"],
    ens_key="12 model ensemble mean",
    figsize=(18, 15),
    stipple_step=5,
    stipple_size=13,
    min_agree_models=8,
    # savepath="fig_var_change_with_stipple.png",
    # dpi=300,
)

### Step 10: Compute Global Means (Supplementary Table S4)

In [ ]:
# Compute the global mean table for selected variables. This calculates the mean values of specified variables across the entire spatial domain 
# for both historical and SSP3-7.0 scenarios, as well as their changes.
glob_mean_table = glob_map_tab.global_mean_table(masked_ds_dict, masked_ds_dict_diff, ['bgws_tran_mean', 'pr_mean', 'mrro_mean', 'tran_mean'], 
                                                 filepath=None)
glob_mean_table

### Step 11: Statistics used in manuscript 

In [ ]:
ds = masked_ds_dict["historical"]["12 model ensemble mean"]

for candidate in ["bgws_tran_mean", "bgws", "hist_bgws"]:
    if candidate in ds:
        bgws = ds[candidate]
        break
else:
    raise KeyError("Could not find BGWS variable. Please set bgws = ds['your_variable_name'].")

# detect latitude name
lat_name = None
for candidate in ["lat", "latitude", "y"]:
    if candidate in bgws.coords:
        lat_name = candidate
        break
if lat_name is None:
    raise KeyError("Could not find latitude coordinate.")

lat = bgws[lat_name]

# area weights ~ cos(latitude)
w_lat = np.cos(np.deg2rad(lat))
# broadcast to full grid
w = xr.broadcast(w_lat, bgws)[0]

def weighted_stats(da, band_mask, weights):
    valid = np.isfinite(da) & band_mask
    w_use = weights.where(valid, 0)

    # area-weighted mean BGWS
    mean_bgws = (da.fillna(0) * w_use).sum() / w_use.sum()

    # area-weighted fraction with BGWS > 0
    frac_positive = w_use.where(da > 0, 0).sum() / w_use.sum() * 100

    return float(frac_positive), float(mean_bgws)

# latitude-band masks
high_lat_mask = lat > 60
mid_lat_mask  = (np.abs(lat) >= 40) & (np.abs(lat) < 60)

# broadcast masks to grid
high_lat_mask = xr.broadcast(high_lat_mask, bgws)[0]
mid_lat_mask  = xr.broadcast(mid_lat_mask, bgws)[0]

high_frac_pos, high_mean = weighted_stats(bgws, high_lat_mask, w)
mid_frac_pos,  mid_mean  = weighted_stats(bgws, mid_lat_mask,  w)

print(f"High latitudes (>60°):")
print(f"  Area-weighted fraction with BGWS > 0: {high_frac_pos:.1f}%")
print(f"  Area-weighted mean BGWS: {high_mean:+.1f}%")

print(f"\nMid-latitudes (40–60°N/S):")
print(f"  Area-weighted fraction with BGWS > 0: {mid_frac_pos:.1f}%")
print(f"  Area-weighted mean BGWS: {mid_mean:+.1f}%")

In [ ]:
# -----------------------------
# 1) Get historical BGWS dataset
# -----------------------------

hist_ds = masked_ds_dict["historical"]["12 model ensemble mean"]

for candidate in ["bgws_tran_mean", "bgws", "hist_bgws"]:
    if candidate in hist_ds:
        bgws_hist = hist_ds[candidate]
        break
else:
    raise KeyError(
        "Could not find historical BGWS variable. "
        "Please set bgws_hist = hist_ds['your_variable_name']."
    )

# -----------------------------
# 2) Get BGWS-change dataset
# -----------------------------
if isinstance(masked_ds_dict_diff, xr.Dataset):
    diff_ds = masked_ds_dict_diff
elif "12 model ensemble mean" in masked_ds_dict_diff:
    diff_ds = masked_ds_dict_diff["12 model ensemble mean"]
elif "ssp370_ff-historical" in masked_ds_dict_diff and "12 model ensemble mean" in masked_ds_dict_diff["ssp370_ff-historical"]:
    diff_ds = masked_ds_dict_diff["ssp370_ff-historical"]["12 model ensemble mean"]
else:
    raise KeyError(
        "Could not infer diff dataset from masked_ds_dict_diff. "
        "Please set diff_ds manually."
    )

for candidate in ["d_bgws", "delta_bgws", "bgws_change", "bgws_diff", "dBGWS", "bgws_tran_mean"]:
    if candidate in diff_ds:
        d_bgws = diff_ds[candidate]
        break
else:
    raise KeyError(
        "Could not find BGWS-change variable. "
        "Please set d_bgws = diff_ds['your_variable_name']."
    )

# -----------------------------
# 3) Detect latitude coordinate
# -----------------------------
lat_name = None
for candidate in ["lat", "latitude", "y"]:
    if candidate in bgws_hist.coords:
        lat_name = candidate
        break
if lat_name is None:
    raise KeyError("Could not find latitude coordinate.")

lat = bgws_hist[lat_name]

# -----------------------------
# 4) Area weights
# -----------------------------
w_lat = np.cos(np.deg2rad(lat))
w = xr.broadcast(w_lat, bgws_hist)[0]

# valid cells where both fields exist
valid = np.isfinite(bgws_hist) & np.isfinite(d_bgws)

# -----------------------------
# 5) Helper for weighted fractions
# -----------------------------
def weighted_fraction(condition, denominator_mask, weights):
    w_den = weights.where(denominator_mask, 0)
    w_num = weights.where(condition & denominator_mask, 0)
    return float((w_num.sum() / w_den.sum()) * 100)

# -----------------------------
# 6) Latitude-band masks
# -----------------------------
high_lat_mask = lat > 60
subtropics_mask = (np.abs(lat) >= 20) & (np.abs(lat) < 40)
tropics_mask = np.abs(lat) < 20

high_lat_mask = xr.broadcast(high_lat_mask, bgws_hist)[0]
subtropics_mask = xr.broadcast(subtropics_mask, bgws_hist)[0]
tropics_mask = xr.broadcast(tropics_mask, bgws_hist)[0]

# -----------------------------
# 7) Conditions of interest
# -----------------------------
hist_blue = bgws_hist > 0
strengthen_blue = hist_blue & (d_bgws > 0)
weaken_blue = hist_blue & (d_bgws < 0)

greater_blue = d_bgws > 0

# -----------------------------
# 8) Fractions
# -----------------------------
# High latitudes: fractions of ALL high-latitude land
high_strength_all = weighted_fraction(
    strengthen_blue, valid & high_lat_mask, w
)
high_weaken_all = weighted_fraction(
    weaken_blue, valid & high_lat_mask, w
)

# High latitudes: fractions WITHIN historically blue high-latitude land only
high_blue_den = valid & high_lat_mask & hist_blue
high_strength_within_blue = weighted_fraction(
    strengthen_blue, high_blue_den, w
)
high_weaken_within_blue = weighted_fraction(
    weaken_blue, high_blue_den, w
)

# Subtropics: all land with greater blue-water share
subtropics_greater_blue = weighted_fraction(
    greater_blue, valid & subtropics_mask, w
)

# Tropics: all land with greater blue-water share
tropics_greater_blue = weighted_fraction(
    greater_blue, valid & tropics_mask, w
)

# -----------------------------
# 9) Print results
# -----------------------------
print("High latitudes (>60°N), fractions of ALL land in band:")
print(f"  Strengthening historical blue-water regime: {high_strength_all:.1f}%")
print(f"  Weakening historical blue-water regime:    {high_weaken_all:.1f}%")

print("\nHigh latitudes (>60°N), fractions WITHIN historically blue land only:")
print(f"  Strengthening historical blue-water regime: {high_strength_within_blue:.1f}%")
print(f"  Weakening historical blue-water regime:    {high_weaken_within_blue:.1f}%")

print("\nSubtropics (20–40°N/S), fraction with greater blue-water share (ΔBGWS > 0):")
print(f"  {subtropics_greater_blue:.1f}%")

print("\nTropics (20°S–20°N), fraction with greater blue-water share (ΔBGWS > 0):")
print(f"  {tropics_greater_blue:.1f}%")

In [ ]:
# Compute the table of percentage changes in BGWS between historical and SSP3-7.0 periods based on their historical BGWS regome. 
change_table = glob_map_tab.percentage_changes_table(masked_ds_dict['historical'], masked_ds_dict_diff['ssp370_ff-historical'], 
                                                     filepath=None)
change_table